# Reconstruction d'images COCO : dataset vs modèle

On vérifie d'abord que **les images du dataset sont correctes** (claires, nettes, avec le masque de segmentation), puis on **entraîne le modèle** et on compare **l'image réelle** avec **l'image générée** par le modèle.

## 0. Imports

In [1]:
# Reconstruction COCO : image réelle vs image générée par le modèle
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from recherche_agi import AnchorNeurons, dynamic_k, visualize_coco_sample

C:\Users\henry\Desktop\workspace\recherche-agi\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Les images du dataset (vérification)

In [2]:
ds = load_dataset('shunk031/cocostuff', 'stuff-thing', split='train',
                  streaming=True, trust_remote_code=True)
# afficher 2 vraies images complètes (image + masque)
for n, x in enumerate(ds):
    if n >= 2: break
    visualize_coco_sample(np.array(x['image']), np.array(x['stuff_map']),
                          x.get('objects', []), out=f'coco_full_{n}.png')
    print(f"image {n}: {np.array(x['image']).shape}")

Using the latest cached version of the module from C:\Users\henry\.cache\huggingface\modules\datasets_modules\datasets\shunk031--cocostuff\0cb0693fa398814099ab0697b71696861200330abf81ab0aff8993edd8964ba2 (last modified on Wed Aug 26 15:26:09 2026) since it couldn't be found locally at shunk031/cocostuff, or remotely on the Hugging Face Hub.


image 0: (480, 640, 3)


image 1: (426, 640, 3)


## 2. Entraîner le modèle sur les patches d'une image

In [3]:
PATCH = 32
# reprendre la 1e image
for n, x in enumerate(ds):
    if n >= 1: break
img = np.array(x['image']); sm = np.array(x['stuff_map'])
H, W = sm.shape
patches = []
for i in range(0, H-PATCH, PATCH):
    for j in range(0, W-PATCH, PATCH):
        p = img[i:i+PATCH, j:j+PATCH].astype(float)/255.0
        patches.append((i, j, p))
d_in = PATCH*PATCH*3
anchors = AnchorNeurons(d_in=d_in, n_neurons=150, seed=0, lr=0.1, use_homeostasis=True)
for i, j, p in patches:
    anchors.learn(p.flatten()/(np.linalg.norm(p)+1e-8), k=1)
print(f"{len(patches)} patches, {len(anchors.W)} neurones formés")

247 patches, 150 neurones formés


## 3. Reconstruction : image réelle vs image générée

In [4]:
recon = np.zeros_like(img, dtype=float)
for i, j, p in patches:
    zn = p.flatten()/(np.linalg.norm(p)+1e-8)
    sims = anchors.W @ zn
    w = int(np.argmax(sims))
    proto = anchors.W[w][:d_in].reshape(PATCH, PATCH, 3)
    proto = (proto - proto.min())/(proto.max()-proto.min()+1e-8)
    recon[i:i+PATCH, j:j+PATCH] = proto*255.0

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(img); axes[0].set_title("Image réelle (dataset)")
axes[0].axis('off')
axes[1].imshow(recon.astype(np.uint8)); axes[1].set_title("Image générée (modèle)")
axes[1].axis('off')
plt.tight_layout(); plt.show()

C:\Users\henry\AppData\Local\Temp\ipykernel_7540\1258779829.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4. Analyse

In [5]:
print("=== ANALYSE : RECONSTRUCTION COCO ===")
print("1. Les images du DATASET sont correctes : claires, nettes, scènes réelles.")
print("   Le masque stuff correspond aux régions (mais les 'things' ne sont pas")
print("   dans la palette stuff -> peut apparaître sombre).")
print("2. La RECONSTRUCTION par le modèle est très dégradée (mosaïque bruitée) :")
print("   - le modèle remplace chaque patch par le PROTOTYPE le plus proche")
print("   - 150 neurones pour des patches 32x32 RGB = capacité insuffisante")
print("   - COCO est bien plus complexe que MNIST (textures, objets, scènes)")
print()
print("=> Résultat honnête : la reconstruction naive par prototypes ne suffit")
print("   pas sur COCO. Les images du dataset sont bonnes, mais le modèle de")
print("   reconstruction est trop simple pour cette complexité.")

=== ANALYSE : RECONSTRUCTION COCO ===
1. Les images du DATASET sont correctes : claires, nettes, scènes réelles.
   Le masque stuff correspond aux régions (mais les 'things' ne sont pas
   dans la palette stuff -> peut apparaître sombre).
2. La RECONSTRUCTION par le modèle est très dégradée (mosaïque bruitée) :
   - le modèle remplace chaque patch par le PROTOTYPE le plus proche
   - 150 neurones pour des patches 32x32 RGB = capacité insuffisante
   - COCO est bien plus complexe que MNIST (textures, objets, scènes)

=> Résultat honnête : la reconstruction naive par prototypes ne suffit
   pas sur COCO. Les images du dataset sont bonnes, mais le modèle de
   reconstruction est trop simple pour cette complexité.
